# 作业 5.1：液体闪烁体的脉冲形状甄别

本作业使用 $2\times2$ 英寸 BC501A 液体闪烁体探测器测得的钚碳中子源数据。信号由 XIA 500 MS/s、14 bit 数字化采集卡记录，因此相邻采样点间隔为 2 ns。

数据文件 [liquidpsd_tree.root](liquidpsd_tree.root) 中的 TTree `wave` 包含 10 000 个 entries，每个 entry 对应一次触发，branch `adc[250]` 保存该次触发的 250 个 ADC samples。波形保留了基线偏置和触发位置的变化，需要在积分前完成基线修正和时间对齐。


<div class="code-language-switch" role="group" aria-label="Code language">
  <span>Code language:</span>
  <button type="button" data-code-language="python" aria-pressed="true">Python / PyROOT</button>
  <button type="button" data-code-language="cpp" aria-pressed="false">ROOT C++</button>
</div>

<style>
.code-language-switch { display:none; gap:.5rem; align-items:center; margin:1rem 0; }
.code-language-switch button { padding:.3rem .8rem; border:1px solid #b8b8b8; border-radius:4px; background:#fff; cursor:pointer; }
.code-language-switch button[aria-pressed="true"] { color:#fff; background:#2f6f9f; border-color:#2f6f9f; }
.pyroot-code-marker { display:none; }
.pyroot-code-marker + .highlight { margin:.5rem 0 1rem; border:1px solid #d5d5d5; background:#f7f7f7; }
.pyroot-code-marker + .highlight pre { margin:0; padding:.75rem 1rem; overflow-x:auto; }
.pyroot-code-cell[hidden], .cpp-code-input[hidden] { display:none !important; }
[id="cell-id=includes"],
[id="cell-id=raw-pyroot"],
[id="cell-id=raw-cpp"] .jp-Cell-inputWrapper,
[id="cell-id=baseline-plot-pyroot"],
[id="cell-id=baseline-plot-cpp"] .jp-Cell-inputWrapper,
[id="cell-id=alignment-plot-pyroot"],
[id="cell-id=alignment-plot-cpp"] .jp-Cell-inputWrapper,
[id="cell-id=psd-pyroot"],
[id="cell-id=psd-cpp"] .jp-Cell-inputWrapper,
[id="cell-id=fom-pyroot"],
[id="cell-id=fom-cpp"] .jp-Cell-inputWrapper,
[id="cell-id=average-pyroot"],
[id="cell-id=average-cpp"] .jp-Cell-inputWrapper { display:none !important; }
</style>

<script>
document.addEventListener("DOMContentLoaded", function () {
  const buttons = document.querySelectorAll(".code-language-switch button");
  const pythonCells = Array.from(document.querySelectorAll(".pyroot-code-marker"))
    .map(function (marker) { return marker.closest(".jp-MarkdownCell"); })
    .filter(Boolean);
  pythonCells.forEach(function (cell) { cell.classList.add("pyroot-code-cell"); });
  const cppInputs = Array.from(document.querySelectorAll(".jp-CodeCell .jp-Cell-inputWrapper"));
  cppInputs.forEach(function (input) { input.classList.add("cpp-code-input"); });

  function selectLanguage(language) {
    pythonCells.forEach(function (cell) { cell.hidden = language !== "python"; });
    cppInputs.forEach(function (input) { input.hidden = language !== "cpp"; });
    buttons.forEach(function (button) {
      button.setAttribute("aria-pressed", String(button.dataset.codeLanguage === language));
    });
  }

  buttons.forEach(function (button) {
    button.addEventListener("click", function () { selectLanguage(button.dataset.codeLanguage); });
  });
  document.querySelector(".code-language-switch").style.display = "flex";
  selectLanguage("python");
});
</script>


## 方法

液体有机闪烁体中，中子主要通过反冲质子沉积能量，gamma 主要通过电子沉积能量。中子脉冲的慢成分占比通常更大，因此可用脉冲尾部积分与总积分的比值进行脉冲形状甄别（PSD）。

处理时先修正每条波形的基线，再统一脉冲的时间位置，最后计算积分和 PSD。


## 数据处理

### 读取 ROOT 文件

读取 `wave` tree，将每个 entry 的 `adc` 保存为一条波形。`SetBranchAddress` 将 branch 连接到数组，`GetEntry` 将指定事例的采样值读入该数组。


In [1]:
#include "TCanvas.h"
#include "TFile.h"
#include "TF1.h"
#include "TFitResultPtr.h"
#include "TGraph.h"
#include "TH1D.h"
#include "TH2D.h"
#include "TLatex.h"
#include "TLegend.h"
#include "TLine.h"
#include "TMultiGraph.h"
#include "TPad.h"
#include "TStyle.h"
#include "TString.h"
#include "TTree.h"
#include <algorithm>
#include <cmath>
#include <iomanip>
#include <iostream>
#include <limits>
#include <numeric>
#include <vector>


<div class="pyroot-code-marker"></div>

~~~python
import ROOT
import math
from array import array
%jsroot on
ROOT.gStyle.SetOptStat(0)

input_file = ROOT.TFile.Open("liquidpsd_tree.root")
wave = input_file.Get("wave")
pulse_count, sample_count = wave.GetEntries(), 250
adc = array("d", [0.0] * sample_count)
wave.SetBranchAddress("adc", adc)
raw_pulses = []
for event in range(pulse_count):
    wave.GetEntry(event)
    raw_pulses.append(list(adc))  # 保存当前波形的一份副本
print(f"{pulse_count} waveforms, {sample_count} samples per waveform")
~~~


In [2]:
//%jsroot on
gStyle->SetOptStat(0);
auto inputFile = TFile::Open("liquidpsd_tree.root");
auto wave = (TTree*)inputFile->Get("wave");
int pulseCount = wave->GetEntries(), sampleCount = 250;
double adc[250];
wave->SetBranchAddress("adc", adc);
std::vector<std::vector<double>> rawPulses;
for (int event = 0; event < pulseCount; ++event) {
    wave->GetEntry(event);
    rawPulses.push_back(std::vector<double>(adc, adc + sampleCount));
}
std::cout << pulseCount << " waveforms, " << sampleCount
          << " samples per waveform" << std::endl;


10000 waveforms, 250 samples per waveform


<div class="pyroot-code-marker"></div>

~~~python
raw20 = raw_pulses[20]
raw30 = raw_pulses[30]
samples = array("d", range(sample_count))
raw_graph20 = ROOT.TGraph(sample_count, samples, array("d", raw20))
raw_graph30 = ROOT.TGraph(sample_count, samples, array("d", raw30))
raw_graph20.SetLineColor(ROOT.kBlue + 1)
raw_graph30.SetLineColor(ROOT.kRed + 1)

raw_graphs = ROOT.TMultiGraph()
raw_graphs.SetTitle("Raw waveforms;sample index;ADC value")
raw_graphs.Add(raw_graph20)
raw_graphs.Add(raw_graph30)
c_raw = ROOT.TCanvas("c_raw_py", "raw waveforms", 800, 480)
raw_graphs.Draw("AL")
c_raw.Draw()
c_raw.SaveAs("raw_waveforms.png")
~~~


In [3]:
auto raw20 = rawPulses[20];
auto raw30 = rawPulses[30];
std::vector<double> samples(sampleCount);
std::iota(samples.begin(), samples.end(), 0.0);
auto rawGraph20 = new TGraph(sampleCount, samples.data(), raw20.data());
auto rawGraph30 = new TGraph(sampleCount, samples.data(), raw30.data());
rawGraph20->SetLineColor(kBlue + 1);
rawGraph30->SetLineColor(kRed + 1);

auto rawGraphs = new TMultiGraph();
rawGraphs->SetTitle("Raw waveforms;sample index;ADC value");
rawGraphs->Add(rawGraph20);
rawGraphs->Add(rawGraph30);
auto cRaw = new TCanvas("cRaw", "raw waveforms", 800, 480);
rawGraphs->Draw("AL");
cRaw->Draw();
cRaw->SaveAs("raw_waveforms.png");


<img src="raw_waveforms.png" alt="两条原始波形" style="width:680px; max-width:100%;" />


### 基线修正

原始波形叠加了约 1665 ADC 的基线偏置，各次记录的偏置略有不同。积分前要减去每条波形自己的基线，否则偏置会随积分门长度累加到电荷中。

用脉冲到来前的 40 个采样点估计基线，再从整条波形中减去它：

$$b=\frac{1}{40}\sum_{i=0}^{39}y_i,\qquad y'_i=y_i-b.$$


<div class="pyroot-code-marker"></div>

~~~python
baselines = []
corrected_pulses = []
for values in raw_pulses:
    baseline = sum(values[:40]) / 40
    baselines.append(baseline)
    corrected_pulses.append([value - baseline for value in values])
print(f"baseline: event 20 = {baselines[20]:.2f}, "
      f"event 30 = {baselines[30]:.2f} ADC")
~~~


In [4]:
std::vector<double> baselines;
std::vector<std::vector<double>> correctedPulses;
for (auto values : rawPulses) {
    double baseline = std::accumulate(values.begin(), values.begin() + 40, 0.0) / 40;
    baselines.push_back(baseline);
    for (double& value : values) value -= baseline;
    correctedPulses.push_back(values);
}
std::cout << std::fixed << std::setprecision(2)
          << "baseline: event 20 = " << baselines[20]
          << ", event 30 = " << baselines[30] << " ADC" << std::endl;


baseline: event 20 = 1666.78, event 30 = 1662.92 ADC


<div class="pyroot-code-marker"></div>

~~~python
c_baseline = ROOT.TCanvas("c_baseline", "baseline correction", 1000, 430)
c_baseline.Divide(2, 1)
baseline_graphs = []
baseline_legends = []
for panel, pulses, title in [
    (1, raw_pulses, "Before baseline correction;sample index;ADC"),
    (2, corrected_pulses, "After baseline correction;sample index;ADC - baseline")
]:
    c_baseline.cd(panel)
    graphs = ROOT.TMultiGraph()
    graphs.SetTitle(title)
    legend = ROOT.TLegend(0.68, 0.72, 0.89, 0.89)
    for event, color in [(20, ROOT.kBlue + 1), (30, ROOT.kRed + 1)]:
        graph = ROOT.TGraph(40, array("d", range(40)), array("d", pulses[event][:40]))
        graph.SetLineColor(color)
        graphs.Add(graph)
        baseline_graphs.append(graph)
        legend.AddEntry(graph, f"event {event}", "l")
    graphs.Draw("AL")
    graphs.GetYaxis().CenterTitle()
    legend.Draw()
    baseline_graphs.append(graphs)
    baseline_legends.append(legend)
c_baseline.Draw()
c_baseline.SaveAs("baseline_correction.png")
~~~


In [5]:
auto cBaseline = new TCanvas("cBaseline", "baseline correction", 1000, 430);
cBaseline->Divide(2, 1);
for (int panel = 1; panel <= 2; ++panel) {
    cBaseline->cd(panel);
    auto graphs = new TMultiGraph();
    graphs->SetTitle(panel == 1
        ? "Before baseline correction;sample index;ADC"
        : "After baseline correction;sample index;ADC - baseline");
    auto legend = new TLegend(0.68, 0.72, 0.89, 0.89);
    for (int event : {20, 30}) {
        auto& values = panel == 1 ? rawPulses[event] : correctedPulses[event];
        auto graph = new TGraph(40, samples.data(), values.data());
        graph->SetLineColor(event == 20 ? kBlue + 1 : kRed + 1);
        graphs->Add(graph);
        legend->AddEntry(graph, Form("event %d", event), "l");
    }
    graphs->Draw("AL");
    graphs->GetYaxis()->CenterTitle();
    legend->Draw();
}
cBaseline->Draw();
cBaseline->SaveAs("baseline_correction.png");


<img src="baseline_correction.png" alt="基线修正前后，放大触发前的40个采样点" style="width:820px; max-width:100%;" />


图中放大了触发前的基线区间。修正后，两条波形都围绕零涨落；减去基线的操作应用于全部 250 个采样点。


### 时间对齐

触发时刻相对于采样时钟会有变化，因此脉冲在各条记录中的位置并不完全相同。若直接使用固定积分门，同一个门的边界会落在不同的脉冲部位，尾部积分随之改变，给 PSD 带来额外的变化。时间对齐使积分门相对于脉冲的位置一致。

本数据的峰顶尖锐，可以用最大采样点定位。取第一条波形的峰位 $k_0$ 为参考，对峰位为 $k_{\max}$ 的波形平移 $\Delta=k_0-k_{\max}$ 个采样点：

$$s_j=y'_{j-\Delta}.$$

$\Delta>0$ 时向右移，$\Delta<0$ 时向左移；空出的采样点补零。峰顶较宽或噪声较大时，最大点容易跳动，可改用前沿的 fast filter 或 constant-fraction 定时。


<div class="pyroot-code-marker"></div>

~~~python
reference = corrected_pulses[0]
alignment_target = reference.index(max(reference))
aligned_pulses, peak_positions = [], []
for values in corrected_pulses:
    peak = values.index(max(values))
    peak_positions.append(peak)
    shift = alignment_target - peak
    aligned = [0.0] * sample_count
    for i in range(sample_count):
        source = i - shift
        if 0 <= source < sample_count:
            aligned[i] = values[source]
    aligned_pulses.append(aligned)

print(f"reference maximum = {alignment_target}")
print(f"event 20: {peak_positions[20]} -> {alignment_target}; "
      f"event 30: {peak_positions[30]} -> {alignment_target}")
print(f"processed {len(aligned_pulses)} waveforms")
~~~


In [6]:
auto reference = correctedPulses[0];
int alignmentTarget = std::max_element(reference.begin(), reference.end()) - reference.begin();
std::vector<std::vector<double>> alignedPulses;
std::vector<int> peakPositions;
for (auto values : correctedPulses) {
    int peak = std::max_element(values.begin(), values.end()) - values.begin();
    peakPositions.push_back(peak);
    int shift = alignmentTarget - peak;
    std::vector<double> aligned(sampleCount, 0.0);
    for (int i = 0; i < sampleCount; ++i) {
        int source = i - shift;
        if (source >= 0 && source < sampleCount) aligned[i] = values[source];
    }
    alignedPulses.push_back(aligned);
}
std::cout << "reference maximum = " << alignmentTarget << "\n"
          << "event 20: " << peakPositions[20] << " -> " << alignmentTarget
          << "; event 30: " << peakPositions[30] << " -> " << alignmentTarget << "\n"
          << "processed " << alignedPulses.size() << " waveforms" << std::endl;


reference maximum = 61
event 20: 58 -> 61; event 30: 61 -> 61
processed 10000 waveforms


<div class="pyroot-code-marker"></div>

~~~python
before20 = ROOT.TGraph(sample_count, samples, array("d", corrected_pulses[20]))
before30 = ROOT.TGraph(sample_count, samples, array("d", corrected_pulses[30]))
after20 = ROOT.TGraph(sample_count, samples, array("d", aligned_pulses[20]))
after30 = ROOT.TGraph(sample_count, samples, array("d", aligned_pulses[30]))
for blue, red in ((before20, before30), (after20, after30)):
    blue.SetLineColor(ROOT.kBlue + 1)
    red.SetLineColor(ROOT.kRed + 1)

c_alignment = ROOT.TCanvas("c_alignment_py", "time alignment", 1000, 430)
c_alignment.Divide(2, 1)
c_alignment.cd(1)
before_graphs = ROOT.TMultiGraph()
before_graphs.SetTitle("Before alignment;sample index;ADC-baseline")
before_graphs.Add(before20)
before_graphs.Add(before30)
before_graphs.Draw("AL")
before_graphs.GetYaxis().CenterTitle()
before_graphs.GetXaxis().SetLimits(45, 90)
before_legend = ROOT.TLegend(0.48, 0.70, 0.88, 0.86)
before_legend.AddEntry(before20, f"event 20: kmax={peak_positions[20]}", "l")
before_legend.AddEntry(before30, f"event 30: kmax={peak_positions[30]}", "l")
before_legend.Draw()
c_alignment.cd(2)
after_graphs = ROOT.TMultiGraph()
after_graphs.SetTitle("After alignment;sample index;ADC-baseline")
after_graphs.Add(after20)
after_graphs.Add(after30)
after_graphs.Draw("AL")
after_graphs.GetYaxis().CenterTitle()
after_graphs.GetXaxis().SetLimits(45, 90)
target_line = ROOT.TLine(alignment_target, 0, alignment_target,
                         1.05 * max(max(aligned_pulses[20]), max(aligned_pulses[30])))
target_line.SetLineStyle(2)
target_line.Draw()
target_label = ROOT.TLatex()
target_label.SetTextSize(0.035)
target_label.DrawLatex(alignment_target + 4,
                       0.92 * max(max(aligned_pulses[20]), max(aligned_pulses[30])),
                       f"k0 = {alignment_target}")
c_alignment.Draw()
c_alignment.SaveAs("time_alignment.png")
~~~


In [7]:
auto before20 = new TGraph(sampleCount, samples.data(), correctedPulses[20].data());
auto before30 = new TGraph(sampleCount, samples.data(), correctedPulses[30].data());
auto after20 = new TGraph(sampleCount, samples.data(), alignedPulses[20].data());
auto after30 = new TGraph(sampleCount, samples.data(), alignedPulses[30].data());
for (auto graph : {before20, after20}) graph->SetLineColor(kBlue + 1);
for (auto graph : {before30, after30}) graph->SetLineColor(kRed + 1);

auto cAlignment = new TCanvas("cAlignment", "time alignment", 1000, 430);
cAlignment->Divide(2, 1);
cAlignment->cd(1);
auto beforeGraphs = new TMultiGraph();
beforeGraphs->SetTitle("Before alignment;sample index;ADC-baseline");
beforeGraphs->Add(before20);
beforeGraphs->Add(before30);
beforeGraphs->Draw("AL");
beforeGraphs->GetYaxis()->CenterTitle();
beforeGraphs->GetXaxis()->SetLimits(45, 90);
auto beforeLegend = new TLegend(0.48, 0.70, 0.88, 0.86);
beforeLegend->AddEntry(before20, Form("event 20: kmax=%d", peakPositions[20]), "l");
beforeLegend->AddEntry(before30, Form("event 30: kmax=%d", peakPositions[30]), "l");
beforeLegend->Draw();
cAlignment->cd(2);
auto afterGraphs = new TMultiGraph();
afterGraphs->SetTitle("After alignment;sample index;ADC-baseline");
afterGraphs->Add(after20);
afterGraphs->Add(after30);
afterGraphs->Draw("AL");
afterGraphs->GetYaxis()->CenterTitle();
afterGraphs->GetXaxis()->SetLimits(45, 90);
const double alignedMaximum = std::max(
    *std::max_element(alignedPulses[20].begin(), alignedPulses[20].end()),
    *std::max_element(alignedPulses[30].begin(), alignedPulses[30].end()));
auto targetLine = new TLine(
    alignmentTarget, 0, alignmentTarget, 1.05 * alignedMaximum);
targetLine->SetLineStyle(2);
targetLine->Draw();
auto targetLabel = new TLatex();
targetLabel->SetTextSize(0.035);
targetLabel->DrawLatex(
    alignmentTarget + 4, 0.92 * alignedMaximum,
    Form("k0 = %d", alignmentTarget));
cAlignment->Draw();
cAlignment->SaveAs("time_alignment.png");


<img src="time_alignment.png" alt="尖锐峰顶在时间对齐前后的局部对照" style="width:820px; max-width:100%;" />


第一条波形的峰位为 61。event 20 的峰位由 58 向右移到 61，event 30 已在 61，不需平移。图中两侧都已减去基线，仅比较时间位置。


### 积分门与脉冲的相对位置

对基线修正并完成时间对齐的波形 $s_i$，分别对整段脉冲和尾部求和：

$$Q_{\rm total}=\sum_{i=t_0}^{t_2-1}s_i,\qquad
Q_{\rm tail}=\sum_{i=t_1}^{t_2-1}s_i,\qquad
PSD=\frac{Q_{\rm tail}}{Q_{\rm total}}.$$

total gate 从上升沿之前开始，覆盖主要脉冲及其尾部；tail gate 从峰后的下降段开始，覆盖其中的慢成分。这里 $t_0,t_1,t_2$ 表示采样点位置，每个采样间隔为 2 ns。


<img src="integration_gates_schematic.png" alt="total gate 与 tail gate 相对于脉冲位置的概念示意图" style="width:680px; max-width:100%;" />


积分门位置示意图。tail gate 是 total gate 内的尾部区间。


### 电荷积分与 PSD 二维图

对每条处理后的波形，在两个积分门内分别求和，再用 $Q_{\rm tail}/Q_{\rm total}$ 计算 PSD。每条波形在二维图中贡献一个事例：左图比较尾部积分与总积分，右图比较尾部占比与总积分。

以下示例取 total gate 为 $[50,240)$、tail gate 为 $[72,240)$，长度分别为 380 ns 和 336 ns。在相近的 $Q_{\rm total}$ 下，中子脉冲的尾部占比更大，因此形成上方条带；gamma 位于下方。


<div class="pyroot-code-marker"></div>

~~~python
gate_start, tail_start, total_end = 50, 72, 240
total_charge = []
psd_value = []
h_tail_total = ROOT.TH2D(
    "h_tail_total_py", "Tail integral;Q_{total} (10^{3} ADC sum);Q_{tail} (10^{3} ADC sum)",
    250, 0, 150, 200, 0, 60
)
h_psd = ROOT.TH2D(
    "h_psd_py", "Pulse-shape discrimination;Q_{total} (10^{3} ADC sum);Q_{tail}/Q_{total}",
    250, 0, 150, 180, 0, 0.5
)

for values in aligned_pulses:
    q_total = sum(values[gate_start:total_end])
    q_tail = sum(values[tail_start:total_end])
    psd = q_tail / q_total if q_total > 0 else float("nan")
    total_charge.append(q_total)
    psd_value.append(psd)
    if math.isfinite(psd):
        h_tail_total.Fill(q_total / 1000.0, q_tail / 1000.0)
        h_psd.Fill(q_total / 1000.0, psd)

c_psd = ROOT.TCanvas("c_psd_py", "PSD maps", 1000, 430)
c_psd.Divide(2, 1)
c_psd.cd(1)
ROOT.gPad.SetLogz()
ROOT.gPad.SetRightMargin(0.16)
h_tail_total.Draw("colz")
c_psd.cd(2)
ROOT.gPad.SetLogz()
ROOT.gPad.SetRightMargin(0.16)
h_psd.Draw("colz")
c_psd.Draw()
c_psd.SaveAs("psd_maps.png")

print(f"total gate: [{gate_start}, {total_end}), {(total_end - gate_start) * 2} ns")
print(f"tail gate: [{tail_start}, {total_end}), {(total_end - tail_start) * 2} ns")
~~~


In [8]:
const int gateStart = 50, tailStart = 72, totalEnd = 240;
std::vector<double> totalCharge(pulseCount);
std::vector<double> psdValue(pulseCount, std::numeric_limits<double>::quiet_NaN());
auto hTailTotal = new TH2D(
    "hTailTotal", "Tail integral;Q_{total} (10^{3} ADC sum);Q_{tail} (10^{3} ADC sum)",
    250, 0, 150, 200, 0, 60);
auto hPsd = new TH2D(
    "hPsd", "Pulse-shape discrimination;Q_{total} (10^{3} ADC sum);Q_{tail}/Q_{total}",
    250, 0, 150, 180, 0, 0.5);

for (int index = 0; index < pulseCount; ++index) {
    const auto& values = alignedPulses[index];
    const double qTotal = std::accumulate(
        values.begin() + gateStart, values.begin() + totalEnd, 0.0);
    const double qTail = std::accumulate(
        values.begin() + tailStart, values.begin() + totalEnd, 0.0);
    totalCharge[index] = qTotal;
    if (qTotal <= 0.0) continue;
    psdValue[index] = qTail / qTotal;
    hTailTotal->Fill(qTotal / 1000.0, qTail / 1000.0);
    hPsd->Fill(qTotal / 1000.0, psdValue[index]);
}

auto cPsd = new TCanvas("cPsd", "PSD maps", 1000, 430);
cPsd->Divide(2, 1);
cPsd->cd(1);
gPad->SetLogz();
gPad->SetRightMargin(0.16);
hTailTotal->Draw("colz");
cPsd->cd(2);
gPad->SetLogz();
gPad->SetRightMargin(0.16);
hPsd->Draw("colz");
cPsd->Draw();
cPsd->SaveAs("psd_maps.png");

std::cout << "total gate: [" << gateStart << ", " << totalEnd << "), "
          << (totalEnd - gateStart) * 2 << " ns\n"
          << "tail gate: [" << tailStart << ", " << totalEnd << "), "
          << (totalEnd - tailStart) * 2 << " ns" << std::endl;


total gate: [50, 240), 380 ns
tail gate: [72, 240), 336 ns


<img src="psd_maps.png" alt="尾部积分和PSD随总积分的二维分布" style="width:840px; max-width:100%;" />


### 固定 $Q_{\rm total}$ 区间计算 FoM

PSD 条带的位置和宽度随脉冲总积分变化，需要在相近的脉冲大小下比较分离效果。选择 $40000\le Q_{\rm total}<60000$ 的事例，将其 PSD 填入一维直方图，再对 gamma 和 neutron 两个峰分别作 Gaussian fit。

用拟合得到的峰位 $\mu$ 和 $FWHM=2.355\sigma$ 计算

$$FoM=\frac{|\mu_n-\mu_\gamma|}{FWHM_n+FWHM_\gamma}.$$

两峰相距越远、各自越窄，FoM 越大。左图标出所选总积分区间，右图给出该区间的 PSD 分布及拟合。


<div class="pyroot-code-marker"></div>

~~~python
charge_low, charge_high = 40000.0, 60000.0
c_fom_region = ROOT.TCanvas("c_fom_region_py", "selected interval", 760, 500)
c_fom_region.SetLogz()
h_psd.Draw("colz")
line_charge_low = ROOT.TLine(charge_low / 1000.0, 0.0, charge_low / 1000.0, 0.5)
line_charge_high = ROOT.TLine(charge_high / 1000.0, 0.0, charge_high / 1000.0, 0.5)
for line in (line_charge_low, line_charge_high):
    line.SetLineColor(ROOT.kRed + 1)
    line.SetLineWidth(2)
    line.Draw()
c_fom_region.Draw()
c_fom_region.SaveAs("fom_region.png")

h_slice = ROOT.TH1D(
    "h_slice_py",
    "40000 #leq Q_{total} < 60000;Q_{tail}/Q_{total};counts",
    160, 0.0, 0.4
)
for charge, psd in zip(total_charge, psd_value):
    if charge_low <= charge < charge_high and math.isfinite(psd):
        h_slice.Fill(psd)

fit_gamma = ROOT.TF1("fit_gamma_py", "gaus", 0.04, 0.12)
fit_neutron = ROOT.TF1("fit_neutron_py", "gaus", 0.19, 0.34)
fit_gamma.SetParameters(h_slice.GetMaximum(), 0.074, 0.009)
fit_neutron.SetParameters(60.0, 0.27, 0.019)
result_gamma = h_slice.Fit(fit_gamma, "LRSQ0")
result_neutron = h_slice.Fit(fit_neutron, "LRSQ0")

mean_gamma = fit_gamma.GetParameter(1)
mean_neutron = fit_neutron.GetParameter(1)
fwhm_gamma = 2.355 * abs(fit_gamma.GetParameter(2))
fwhm_neutron = 2.355 * abs(fit_neutron.GetParameter(2))
fom = abs(mean_neutron - mean_gamma) / (fwhm_neutron + fwhm_gamma)

fit_gamma.SetLineColor(ROOT.kBlue + 1)
fit_neutron.SetLineColor(ROOT.kRed + 1)
c_fom = ROOT.TCanvas("c_fom_py", "FoM", 760, 500)
h_slice.Draw("E")
fit_gamma.Draw("same")
fit_neutron.Draw("same")
legend_fom = ROOT.TLegend(0.62, 0.68, 0.87, 0.85)
legend_fom.AddEntry(fit_gamma, "gamma", "l")
legend_fom.AddEntry(fit_neutron, "neutron", "l")
legend_fom.Draw()
label_fom = ROOT.TLatex()
label_fom.SetNDC()
label_fom.SetTextSize(0.04)
label_fom.DrawLatex(0.16, 0.80, f"FoM = {fom:.2f}")
c_fom.Draw()
c_fom.SaveAs("fom_projection.png")

print(f"gamma: mu={mean_gamma:.4f}, FWHM={fwhm_gamma:.4f}")
print(f"neutron: mu={mean_neutron:.4f}, FWHM={fwhm_neutron:.4f}")
print(f"FoM = {fom:.3f}; fit status = {int(result_gamma)}/{int(result_neutron)}")
~~~


In [9]:
const double chargeLow = 40000.0;
const double chargeHigh = 60000.0;
auto cFomRegion = new TCanvas("cFomRegion", "selected interval", 760, 500);
cFomRegion->SetLogz();
hPsd->Draw("colz");
auto lineChargeLow = new TLine(chargeLow / 1000.0, 0.0, chargeLow / 1000.0, 0.5);
auto lineChargeHigh = new TLine(chargeHigh / 1000.0, 0.0, chargeHigh / 1000.0, 0.5);
for (auto line : {lineChargeLow, lineChargeHigh}) {
    line->SetLineColor(kRed + 1);
    line->SetLineWidth(2);
    line->Draw();
}
cFomRegion->Draw();
cFomRegion->SaveAs("fom_region.png");

auto hSlice = new TH1D(
    "hSlice", "40000 #leq Q_{total} < 60000;Q_{tail}/Q_{total};counts",
    160, 0.0, 0.4);
for (int index = 0; index < pulseCount; ++index) {
    if (totalCharge[index] >= chargeLow && totalCharge[index] < chargeHigh
        && std::isfinite(psdValue[index])) {
        hSlice->Fill(psdValue[index]);
    }
}

auto fitGamma = new TF1("fitGamma", "gaus", 0.04, 0.12);
auto fitNeutron = new TF1("fitNeutron", "gaus", 0.19, 0.34);
fitGamma->SetParameters(hSlice->GetMaximum(), 0.074, 0.009);
fitNeutron->SetParameters(60.0, 0.27, 0.019);
TFitResultPtr resultGamma = hSlice->Fit(fitGamma, "LRSQ0");
TFitResultPtr resultNeutron = hSlice->Fit(fitNeutron, "LRSQ0");

const double meanGamma = fitGamma->GetParameter(1);
const double meanNeutron = fitNeutron->GetParameter(1);
const double fwhmGamma = 2.355 * std::abs(fitGamma->GetParameter(2));
const double fwhmNeutron = 2.355 * std::abs(fitNeutron->GetParameter(2));
const double fom = std::abs(meanNeutron - meanGamma)
    / (fwhmNeutron + fwhmGamma);

fitGamma->SetLineColor(kBlue + 1);
fitNeutron->SetLineColor(kRed + 1);
auto cFom = new TCanvas("cFom", "FoM", 760, 500);
hSlice->Draw("E");
fitGamma->Draw("same");
fitNeutron->Draw("same");
auto legendFom = new TLegend(0.62, 0.68, 0.87, 0.85);
legendFom->AddEntry(fitGamma, "gamma", "l");
legendFom->AddEntry(fitNeutron, "neutron", "l");
legendFom->Draw();
auto labelFom = new TLatex();
labelFom->SetNDC();
labelFom->SetTextSize(0.04);
labelFom->DrawLatex(0.16, 0.80, Form("FoM = %.2f", fom));
cFom->Draw();
cFom->SaveAs("fom_projection.png");

std::cout << std::fixed << std::setprecision(4)
          << "gamma: mu=" << meanGamma << ", FWHM=" << fwhmGamma << "\n"
          << "neutron: mu=" << meanNeutron << ", FWHM=" << fwhmNeutron << "\n"
          << std::setprecision(3) << "FoM = " << fom << "; fit status = "
          << static_cast<int>(resultGamma) << "/"
          << static_cast<int>(resultNeutron) << std::endl;


gamma: mu=0.0760, FWHM=0.0234
neutron: mu=0.2862, FWHM=0.0472
FoM = 2.977; fit status = 0/0


<div style="display:flex; gap:1rem; align-items:flex-start;">
  <img src="fom_region.png" alt="selected Q total interval" style="width:47%;" />
  <img src="fom_projection.png" alt="PSD projection and Gaussian fits" style="width:47%;" />
</div>

该示例得到 $FoM\approx2.98$。这个数值只对应所选 $Q_{\rm total}$ 区间和积分门；不能直接当作整个二维分布的单一性能指标。


### 平均脉冲与积分门

单条波形既有幅度差异，也有统计涨落。为了看清两类脉冲的形状差异，在上面同一个 $Q_{\rm total}$ 区间内，用两个 PSD 峰位的中点将事例分为两组。先把每条已对齐的波形除以自己的 $Q_{\rm total}$，再在每个采样点分别求平均：

$$\bar u_a(i)=\frac{1}{N_a}\sum_{j\in a}\frac{s_{j,i}}{Q_{{\rm total},j}},
\qquad a=\gamma,\ n.$$

归一化使每条波形在 total gate 内的面积都为 1，避免大脉冲主导平均值；逐点平均减小随机涨落，留下较清楚的脉冲形状。上图比较两类平均波形，下图给出差值 $D_i=\bar u_n(i)-\bar u_\gamma(i)$。

差值为正的尾部表示中子脉冲在这里占有更大的面积比例。将差值在 tail gate 内求和，恰好得到两组事例平均 PSD 的差：

$$\sum_{i=t_1}^{t_2-1}D_i=\overline{PSD}_n-\overline{PSD}_\gamma.$$

这说明了尾部积分为什么能够甄别两类脉冲。平均波形帮助选择尾部区间，但门也不能无限延长：脉冲回到基线后的采样点主要增加噪声，最终门宽仍通过 FoM 比较。


<div class="pyroot-code-marker"></div>

~~~python
split_psd = 0.5 * (mean_gamma + mean_neutron)
average_gamma = [0.0] * sample_count
average_neutron = [0.0] * sample_count
number_gamma = number_neutron = 0

for values, charge, psd in zip(aligned_pulses, total_charge, psd_value):
    if not charge_low <= charge < charge_high:
        continue
    target = average_gamma if psd < split_psd else average_neutron
    for i, value in enumerate(values):
        target[i] += value / charge
    if psd < split_psd:
        number_gamma += 1
    else:
        number_neutron += 1

average_gamma = [value / number_gamma for value in average_gamma]
average_neutron = [value / number_neutron for value in average_neutron]
difference = [n - g for n, g in zip(average_neutron, average_gamma)]
g_average_gamma = ROOT.TGraph(sample_count, samples, array("d", average_gamma))
g_average_neutron = ROOT.TGraph(sample_count, samples, array("d", average_neutron))
g_difference = ROOT.TGraph(sample_count, samples, array("d", difference))
g_average_gamma.SetLineColor(ROOT.kBlue + 1)
g_average_neutron.SetLineColor(ROOT.kRed + 1)
g_difference.SetLineColor(ROOT.kBlack)

c_average = ROOT.TCanvas("c_average_py", "average pulses", 800, 650)
c_average.Divide(1, 2)
c_average.cd(1)
average_graphs = ROOT.TMultiGraph()
average_graphs.SetTitle("Normalized average pulses;sample index;average amplitude")
average_graphs.Add(g_average_gamma)
average_graphs.Add(g_average_neutron)
average_graphs.Draw("AL")
legend_average = ROOT.TLegend(0.68, 0.68, 0.87, 0.84)
legend_average.AddEntry(g_average_gamma, "gamma", "l")
legend_average.AddEntry(g_average_neutron, "neutron", "l")
legend_average.Draw()
c_average.cd(2)
g_difference.SetTitle("neutron - gamma;sample index;amplitude difference")
g_difference.Draw("AL")
zero_difference = ROOT.TLine(0.0, 0.0, sample_count - 1.0, 0.0)
zero_difference.SetLineStyle(2)
zero_difference.Draw()
c_average.Draw()
c_average.SaveAs("average_pulses.png")
print(f"averaged {number_gamma} gamma-like and {number_neutron} neutron-like pulses")
~~~


In [10]:
const double splitPsd = 0.5 * (meanGamma + meanNeutron);
std::vector<double> averageGamma(sampleCount, 0.0);
std::vector<double> averageNeutron(sampleCount, 0.0);
int numberGamma = 0;
int numberNeutron = 0;
for (int index = 0; index < pulseCount; ++index) {
    const double charge = totalCharge[index];
    if (charge < chargeLow || charge >= chargeHigh) continue;
    auto& average = psdValue[index] < splitPsd ? averageGamma : averageNeutron;
    for (int i = 0; i < sampleCount; ++i) {
        average[i] += alignedPulses[index][i] / charge;
    }
    if (psdValue[index] < splitPsd) ++numberGamma;
    else ++numberNeutron;
}
std::vector<double> difference(sampleCount);
for (int i = 0; i < sampleCount; ++i) {
    averageGamma[i] /= numberGamma;
    averageNeutron[i] /= numberNeutron;
    difference[i] = averageNeutron[i] - averageGamma[i];
}

auto gAverageGamma = new TGraph(sampleCount, samples.data(), averageGamma.data());
auto gAverageNeutron = new TGraph(sampleCount, samples.data(), averageNeutron.data());
auto gDifference = new TGraph(sampleCount, samples.data(), difference.data());
gAverageGamma->SetLineColor(kBlue + 1);
gAverageNeutron->SetLineColor(kRed + 1);
gDifference->SetLineColor(kBlack);

auto cAverage = new TCanvas("cAverage", "average pulses", 800, 650);
cAverage->Divide(1, 2);
cAverage->cd(1);
auto averageGraphs = new TMultiGraph();
averageGraphs->SetTitle("Normalized average pulses;sample index;average amplitude");
averageGraphs->Add(gAverageGamma);
averageGraphs->Add(gAverageNeutron);
averageGraphs->Draw("AL");
auto legendAverage = new TLegend(0.68, 0.68, 0.87, 0.84);
legendAverage->AddEntry(gAverageGamma, "gamma", "l");
legendAverage->AddEntry(gAverageNeutron, "neutron", "l");
legendAverage->Draw();
cAverage->cd(2);
gDifference->SetTitle("neutron - gamma;sample index;amplitude difference");
gDifference->Draw("AL");
auto zeroDifference = new TLine(0.0, 0.0, sampleCount - 1.0, 0.0);
zeroDifference->SetLineStyle(2);
zeroDifference->Draw();
cAverage->Draw();
cAverage->SaveAs("average_pulses.png");
std::cout << "averaged " << numberGamma << " gamma-like and "
          << numberNeutron << " neutron-like pulses" << std::endl;


averaged 267 gamma-like and 588 neutron-like pulses


<img src="average_pulses.png" alt="面积归一后的平均脉冲及中子减gamma差值" style="width:680px; max-width:100%;" />


分组来自同一 PSD 变量，这里用于解释脉冲形状差异。


## 作业要求

1. 观察若干原始波形，确定基线区间，并比较基线修正和时间对齐前后的波形。
2. 参照上述结果，处理全部 10 000 条波形，绘制 $Q_{\rm tail}$–$Q_{\rm total}$ 和 $PSD$–$Q_{\rm total}$ 二维关联图，辨认 gamma 与 neutron 两条带。
3. 在两条带均有足够事例的一个 $Q_{\rm total}$ 区间内投影 PSD，拟合两个 peak 并计算 FoM。
4. 根据平均脉冲形状选择合理的 $t_0$ 和 $t_2$，再比较若干 $t_1$；用 FoM 给出所选 tail gate 和 total gate，同时以采样点和 ns 报告范围。
5. 用当前 PSD 结果暂时选择 gamma 与 neutron 事例，分别作总面积归一后求平均脉冲。比较两条平均波形及其差值，说明积分门为何能区分两类事例。这里的分类来自同一 PSD 变量，只用于解释波形差异，不作为独立验证。

进度对应第 5 章。
